# 🗾 Japan Public Open Data in 3 Minutes with DuckDB & Python
### Analyze Japanese Government Open Data (Procurement, Corporate, Land Prices, Healthcare) Directly in Browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/japan-opendata-lab/japan-official-data-megabank/blob/main/notebooks/japan_open_data_in_3_minutes.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Showcase%20Dataset-blue)](https://huggingface.co/datasets/japan-opendata/japan-official-data-megabank-showcase)
[![GitHub](https://img.shields.io/badge/GitHub-japan--official--data--megabank-black?logo=github)](https://github.com/japan-opendata-lab/japan-official-data-megabank)
[![RapidAPI](https://img.shields.io/badge/RapidAPI-Megabank%20API-orange)](https://rapidapi.com/organization/japan-opendata-lab)

---

## 💡 What is this?
Japan's public sector publishes valuable data, but it is notoriously hard to use: messy Excel sheets, mixed full-width characters, non-standard date formats (Wareki/Japanese Eras), and shifted encodings (Shift-JIS).

**Japan Open Data Lab** has normalized **155+ Japanese official datasets (over 10.16 million records)** into US-Standard 24-column Parquet tables with universal keys (`corporate_number` and `municipality_code`).

In this 3-minute tutorial, you will:
1. Query remote Parquet datasets directly from Hugging Face using **DuckDB** in-memory (Zero download / Zero setup).
2. Analyze top government procurement vendors (GEPS).
3. Explore 44-year historical land price trends (MLIT).
4. Perform cross-domain multi-table SQL joins via universal corporate keys.

## ⚙️ Step 0: Setup (DuckDB & Plotly)
Colab has most data libraries pre-installed. Let's make sure `duckdb` is ready.

In [ ]:
!pip install duckdb plotly pandas -q

import duckdb
import pandas as pd
import plotly.express as px

con = duckdb.connect()
HF_BASE = "https://huggingface.co/datasets/japan-opendata/japan-official-data-megabank-showcase/resolve/main/data"
print("DuckDB Version:", duckdb.__version__)
print("Setup completed successfully!")

---
## 🏛️ Recipe 1: Japan Government Procurement Analysis (GEPS)
**Source**: Digital Agency / GEPS (Government Electronic Procurement System)

Let's find the **Top 10 Suppliers by Total Award Value** directly querying remote Parquet files without downloading them.

In [ ]:
procurement_url = f"{HF_BASE}/japan_public_procurement_sample_5k.parquet"

query_proc = f"""
    SELECT 
        legal_name AS vendor_name,
        corporate_number,
        COUNT(*) AS total_contracts,
        ROUND(SUM(capital_amount_jpy) / 1e8, 2) AS total_award_100m_yen
    FROM read_parquet('{procurement_url}')
    WHERE legal_name IS NOT NULL
    GROUP BY legal_name, corporate_number
    ORDER BY total_award_100m_yen DESC
    LIMIT 10
"""

df_proc = con.execute(query_proc).df()
display(df_proc)

# Interactive Bar Chart
fig = px.bar(
    df_proc,
    x='total_award_100m_yen',
    y='vendor_name',
    orientation='h',
    title='Top 10 Public Procurement Awardees (100 Million JPY)',
    labels={'total_award_100m_yen': 'Total Award (億円)', 'vendor_name': 'Company'},
    color='total_award_100m_yen',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=450)
fig.show()

---
## 🏙️ Recipe 2: Land Price Benchmark Points (MLIT 44-Year Archive)
**Source**: Ministry of Land, Infrastructure, Transport and Tourism (MLIT)

Explore the most valuable commercial and residential land evaluation points in Japan.

In [ ]:
land_url = f"{HF_BASE}/japan_land_prices_sample_5k.parquet"

query_land = f"""
    SELECT 
        area_name_local AS location,
        city,
        current_value AS land_price_jpy_per_sqm,
        ROUND(current_value / 1e4, 1) AS price_10k_yen_sqm,
        yoy_change_pct AS yoy_growth_pct,
        event_date AS valuation_date
    FROM read_parquet('{land_url}')
    WHERE current_value IS NOT NULL
    ORDER BY current_value DESC
    LIMIT 10
"""

df_land = con.execute(query_land).df()
display(df_land)

---
## 🔗 Recipe 3: Cross-Domain Multi-Table SQL JOIN
### Linking Public Procurement with the 13-Digit Corporate Registry
Every normalized table in our megabank shares standard identifiers. Here we JOIN **Government Procurement** with the **National Tax Agency Corporate Registry** on `corporate_number` to inspect where winning contractors are headquartered.

In [ ]:
corporate_url = f"{HF_BASE}/japan_corporate_invoices_sample_5k.parquet"

query_join = f"""
    SELECT 
        p.legal_name AS vendor_name,
        p.corporate_number,
        c.prefecture_en AS hq_prefecture,
        c.city_ja AS hq_city,
        COUNT(p.primary_name) AS total_bids_won,
        ROUND(SUM(p.capital_amount_jpy) / 1e6, 2) AS award_value_million_yen
    FROM read_parquet('{procurement_url}') p
    INNER JOIN read_parquet('{corporate_url}') c
        ON p.corporate_number = c.corporate_number
    WHERE p.corporate_number IS NOT NULL
    GROUP BY p.legal_name, p.corporate_number, c.prefecture_en, c.city_ja
    ORDER BY award_value_million_yen DESC
    LIMIT 10
"""

df_join = con.execute(query_join).df()
display(df_join)

---
## 🏥 Recipe 4: Healthcare Facilities Distribution by Prefecture
**Source**: MHLW (Ministry of Health, Labour and Welfare)

Inspect Japan's nationwide healthcare facilities distributed across prefectures.

In [ ]:
medical_url = f"{HF_BASE}/japan_medical_hospital_sample_5k.parquet"

query_med = f"""
    SELECT 
        prefecture,
        COUNT(*) AS total_facilities
    FROM read_parquet('{medical_url}')
    WHERE prefecture IS NOT NULL
    GROUP BY prefecture
    ORDER BY total_facilities DESC
    LIMIT 10
"""

df_med = con.execute(query_med).df()
display(df_med)

---
## 🚀 Next Steps: Access Full 10.16M+ Records in Production

The Hugging Face datasets are lightweight showcase samples (2,000 to 5,000 records per category).

For production applications, algorithmic trading, and commercial LLM workflows, access the **Japan Official Data Megabank API** on RapidAPI Hub:

| Feature | Showcase Sample (Hugging Face) | Production API (RapidAPI Hub) |
| :--- | :---: | :---: |
| **Total Records** | ~35,000 records | **10.16+ Million Records** |
| **Historical Range** | Sample subset | **Up to 44 Years (1983 - 2026)** |
| **Raw SQL Execution** | Local DuckDB | **Direct Server-Side DuckDB SQL (`POST /api/v1/query`)** |
| **Automated Updates** | Static sample | **Regular Official Synchronizations** |
| **Pricing** | **Free ($0)** | **BASIC ($0) / PRO ($99) / ULTRA ($299) / MEGA ($499)** |

### ⚡ Quickstart with RapidAPI (Python)
```python
# Sign up for a free key at: https://rapidapi.com/organization/japan-opendata-lab
import requests

url = "https://japan-demographics-api.onrender.com/api/v1/procurement/awards"
headers = {
    "X-RapidAPI-Key": "YOUR_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "japan-demographics-api.onrender.com"
}
params = {"limit": 10, "min_amount": 100000000}
response = requests.get(url, headers=headers, params=params)
print(response.json())
```

### 🤖 Using Claude Desktop / Cursor / Windsurf?
Connect Claude directly to this megabank using our open-source MCP Server:
👉 **GitHub**: [japan-opendata-lab/japan-data-mcp](https://github.com/japan-opendata-lab/japan-data-mcp)
👉 **Smithery (100/100 Quality Certified)**: [Japan Public Open Data MCP Server](https://smithery.ai/)

---
**Japan Open Data Lab** — Democratizing Japanese Public Open Data for the World.